# Phase 3.5 — Clean Layer Validation & QA

Read-only checks on `clean_insider_transactions` before market data and forward returns.

Run validation first:

```bash
python -m src.analytics.validate_clean_layer
```

## 1. Why validation matters

The clean layer encodes taxonomies, quality flags, and the MVP alpha universe. Validation confirms internal consistency so we do not propagate bad rows into price joins or return studies.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXPORT_DIR = PROJECT_ROOT / "data" / "exports" / "validation"
print(f"Exports: {EXPORT_DIR}")

## 2. Row count summary

In [ ]:
display(pd.read_csv(EXPORT_DIR / "row_count_summary.csv"))

## 3. MVP universe integrity

In [ ]:
violations = pd.read_csv(EXPORT_DIR / "mvp_integrity_violations.csv")
display(violations)
bad = violations[violations["violation_count"] > 0]
print(f"Violation types with count > 0: {len(bad)}")

## 4. Survival funnel

In [ ]:
from IPython.display import Image

display(pd.read_csv(EXPORT_DIR / "filter_survival_funnel.csv"))
chart = EXPORT_DIR / "charts" / "filter_survival_funnel.png"
if chart.exists():
    display(Image(filename=str(chart)))

## 5. Transaction code sanity

In [ ]:
codes = pd.read_csv(EXPORT_DIR / "transaction_code_distribution_clean.csv")
display(codes[codes["universe"] == "mvp"].head(20))

## 6. Security type sanity

In [ ]:
sec = pd.read_csv(EXPORT_DIR / "security_type_distribution_clean.csv")
display(sec.groupby("universe").head(15))

## 7. Role distribution

In [ ]:
roles = pd.read_csv(EXPORT_DIR / "role_distribution_clean.csv")
display(roles[roles["universe"] == "mvp"].head(20))

## 8. Filing lag sanity

In [ ]:
display(pd.read_csv(EXPORT_DIR / "filing_lag_clean_summary.csv"))
lag_chart = EXPORT_DIR / "charts" / "filing_lag_distribution_mvp.png"
if lag_chart.exists():
    display(Image(filename=str(lag_chart)))

## 9. Transaction value sanity

In [ ]:
display(pd.read_csv(EXPORT_DIR / "transaction_value_summary_mvp.csv"))
val_chart = EXPORT_DIR / "charts" / "transaction_value_distribution_mvp.png"
if val_chart.exists():
    display(Image(filename=str(val_chart)))

## 10. Yearly activity

In [ ]:
yearly = pd.read_csv(EXPORT_DIR / "yearly_mvp_activity.csv")
display(yearly.tail(10))
yr_chart = EXPORT_DIR / "charts" / "yearly_mvp_rows.png"
if yr_chart.exists():
    display(Image(filename=str(yr_chart)))

## 11. Ticker quality (placeholders and multi-symbol)

In [ ]:
for f in ("ticker_quality_summary.csv", "invalid_ticker_samples.csv"):
    p = EXPORT_DIR / f
    if p.exists():
        print(f"--- {f} ---")
        display(pd.read_csv(p).head(20))

## 12. Stale filing summary

In [ ]:
stale_path = EXPORT_DIR / "stale_filing_summary.csv"
if stale_path.exists():
    display(pd.read_csv(stale_path))
by_year = EXPORT_DIR / "stale_filing_by_year_mvp.csv"
if by_year.exists():
    display(pd.read_csv(by_year).tail(10))

## 13. Pre-source transaction dates

In [ ]:
pre_path = EXPORT_DIR / "pre_source_transaction_date_summary.csv"
if pre_path.exists():
    display(pd.read_csv(pre_path))

## 14. Unknown role refinement candidates

In [ ]:
top_unknown = EXPORT_DIR / "top_unknown_roles_for_mapping.csv"
if top_unknown.exists():
    display(pd.read_csv(top_unknown).head(30))
else:
    print("Run build or validation to generate unknown role exports.")

## 15. Ticker readiness (top tickers)

In [ ]:
display(pd.read_csv(EXPORT_DIR / "ticker_readiness_summary.csv"))
display(pd.read_csv(EXPORT_DIR / "top_tickers_mvp.csv").head(20))

## 16. Final PASS / WARN / FAIL summary

| Severity | Meaning |
|----------|--------|
| **PASS** | Check met; safe to proceed |
| **WARN** | Review recommended; may not block Phase 4 |
| **FAIL** | Must fix clean layer or config before market data |

Validation does **not** mutate raw or clean tables.

In [ ]:
summary = pd.read_csv(EXPORT_DIR / "validation_summary.csv")
display(summary["severity"].value_counts())
display(summary[summary["severity"] != "PASS"])